In [1]:
# Importación de librerías
import mne
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
import seaborn as sns
from scipy.stats import shapiro, ttest_rel, wilcoxon
from IPython.display import display

In [2]:
# Verificación de carpetas completas (sujetos completos)

# Carpeta de datos
ruta = 'ds004362'

# Detectar carpetas de sujetos
sujetos = [s for s in os.listdir(ruta) if s.startswith('sub-')]

print(sujetos)

['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005', 'sub-006', 'sub-007', 'sub-008', 'sub-009', 'sub-010', 'sub-011', 'sub-012', 'sub-013', 'sub-014', 'sub-015', 'sub-016', 'sub-017', 'sub-018', 'sub-019', 'sub-020', 'sub-021', 'sub-022', 'sub-023', 'sub-024', 'sub-025', 'sub-026', 'sub-027', 'sub-028', 'sub-029', 'sub-030', 'sub-031', 'sub-032', 'sub-033', 'sub-034', 'sub-035', 'sub-036', 'sub-037', 'sub-038', 'sub-039', 'sub-040', 'sub-041', 'sub-042', 'sub-043', 'sub-044', 'sub-045', 'sub-046', 'sub-047', 'sub-048', 'sub-049', 'sub-050', 'sub-051', 'sub-052', 'sub-053', 'sub-054', 'sub-055', 'sub-056', 'sub-057', 'sub-058', 'sub-059', 'sub-060', 'sub-061', 'sub-062', 'sub-063', 'sub-064', 'sub-065', 'sub-066', 'sub-067', 'sub-068', 'sub-069', 'sub-070', 'sub-071', 'sub-072', 'sub-073', 'sub-074', 'sub-075', 'sub-076', 'sub-077', 'sub-078', 'sub-079', 'sub-080', 'sub-081', 'sub-082', 'sub-083', 'sub-084', 'sub-085', 'sub-086', 'sub-087', 'sub-088', 'sub-089', 'sub-090', 'sub-091'

In [3]:
# Función filtrado de señales

def filtrar(senal):
    s_filt = senal.copy().filter(8., 30., fir_window='hamming', verbose=False)

    return s_filt

In [4]:
# Función para calcular PSD por época

def calcular_psd(epocas, sfreq, canales, nombre_cond, sujeto):

    bandas = {
        "mu": (8, 12),
        "beta": (13, 30)
    }

    # PSD para cada época
    # salida: (n_epocas, n_canales, n_freqs)
    psd, freqs = mne.time_frequency.psd_array_welch(epocas, sfreq=sfreq,
        fmin=1, fmax=40, verbose=False)

    filas = []

    # Recorrer cada época
    for epoca_idx in range(psd.shape[0]):

        fila = {
            "sujeto": sujeto,
            "tarea": nombre_cond,
            "epoca": epoca_idx
        }

        # PSD de la época actual
        psd_epoca = psd[epoca_idx]

        # Calcular potencia por banda
        for banda, (fmin, fmax) in bandas.items():

            idx = (freqs >= fmin) & (freqs <= fmax)

            # Promedio de frecuencias de la banda
            potencia_banda = psd_epoca[:, idx].mean(axis=1)

            # Guardar por canal
            for i, ch in enumerate(canales):
                fila[f"{banda}_{ch}"] = potencia_banda[i]

        filas.append(fila)

    return pd.DataFrame(filas)

In [5]:
# Función principal: Procesamiento de señales EEG

def procesamiento_eeg(ruta, inicio, fin, archivo_salida="resultados_psd.csv"):
    
    sujetos = [s for s in os.listdir(ruta) if s.startswith('sub-')]
    sujetos_rango = sujetos[inicio:fin]
    
    # Ver si ya existe archivo previo
    if os.path.exists(archivo_salida):
        df_total = pd.read_csv(archivo_salida)
        sujetos_procesados = set(df_total["sujeto"].unique())
        print(f"Archivo existente cargado ({len(sujetos_procesados)} sujetos ya procesados)")
    else:
        df_total = pd.DataFrame()
        sujetos_procesados = set()
    
    for sujeto in sujetos_rango:
        
        if sujeto in sujetos_procesados:
            print(f"{sujeto} ya procesado, se omite")
            continue
        
        print(f"Procesando {sujeto}")
        
        ruta_sujeto = os.path.join(ruta, sujeto, "eeg")
        archivos = [f for f in os.listdir(ruta_sujeto) if f.endswith('.set')]
        
        runs_imag = ['run-4', 'run-8', 'run-12']
        
        raws_imag = []

        for archivo in archivos:
            ruta_archivo = os.path.join(ruta_sujeto, archivo)

            # Seleccionar solo los runs deseados
        
            if any(run in archivo for run in runs_imag):
                raw = mne.io.read_raw_eeglab(ruta_archivo, preload=True, verbose=False)
                raws_imag.append(raw)

        # Concatenar
        raw_imag = mne.concatenate_raws(raws_imag)


        # 2. Selección de canales de importancia
        canales = ['C3', 'C4', 'Cz']
        raw_imag.pick_channels(canales)
        
        # Filtrado
        imag_filtrado = filtrar(raw_imag)
        
        # Eventos
        events_i, event_id_i = mne.events_from_annotations(imag_filtrado, verbose=False)
        
        # Épocas
        epochs_imag = mne.Epochs(imag_filtrado, events_i, event_id=event_id_i, tmin=0, tmax=4, 
                                    baseline=None, preload=True, verbose=False)
        
        sfreq = raw_imag.info['sfreq']
        
        # Separación de las 4 condiciones

        condiciones = {
            "Reposo": epochs_imag['TASK1T0'],
            "Izquierda_Imagineria": epochs_imag['TASK2T1'],
            "Derecha_Imagineria": epochs_imag['TASK2T2']
        }
        
        # Cálculo del PSD

        for cond, ep in condiciones.items():
            datos = ep.get_data()
            df_psd = calcular_psd(datos, sfreq, canales, cond, sujeto)
            archivo_out = "resultados_psd.csv"
            df_psd.to_csv(archivo_out, mode='a', header=not os.path.exists(archivo_out), index=False)
        
        print(f"{sujeto} procesado")


    print("Sujetos procesados - Proceso finalizado")